# What's In My Thattu — Food Recognition Model Training

Trains a food recognition model on **multiple combined datasets** for broad food coverage
(Indian, Asian, Western, and more), then exports a `.tflite` for the Android app.

## Before You Run — Add These Datasets

Click **"+ Add Input"** (right sidebar) and search for these datasets:

| Dataset | Kaggle Path | Classes | Why |
|---------|-------------|---------|-----|
| **Indian Food Images** | `iamsouravbanerjee/indian-food-images-dataset` | 80 | Dosa, biryani, samosa, etc. |
| **MAFood-121** | `theviz/mafood121` | 121 | Multi-cuisine (Indian, Thai, Chinese, Western) |
| **UECFood-256** | `rkuo2000/uecfood256` | 256 | Japanese, Asian food |
| **Food-101** | `dansbecker/food-101` | 101 | Western food baseline |
| **Massive Indian Food** | `anshulmehtakaggl/themassiveindianfooddataset` | 15 | High-quality Indian staples |

You can add **any or all** of these — the notebook auto-discovers whatever you attach.
More datasets = broader food recognition.

## Settings Required
- **Accelerator**: GPU T4 x2 (Settings → Accelerator)
- **Internet**: ON (Settings → Internet)
- **Persistence**: Files only (recommended)

## 1. Environment Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tflite-support"])

import tensorflow as tf
import numpy as np
import os, shutil, json, glob
from pathlib import Path
from collections import defaultdict

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"GPUs: {len(gpus)} available" if gpus else "WARNING: No GPU! Enable in Settings -> Accelerator")

## 2. Configuration

In [ ]:
import os

# === Paths ===
INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"
MERGED_DIR = os.path.join(OUTPUT_DIR, "merged_dataset")
TFLITE_MODEL_PATH = os.path.join(OUTPUT_DIR, "whats_in_my_thattu_v2.tflite")
LABEL_MAP_PATH = os.path.join(OUTPUT_DIR, "labels.txt")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# === Image ===
IMAGE_SIZE = 224
CHANNELS = 3
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# === Training ===
BATCH_SIZE = 32
PHASE1_EPOCHS = 25          # Transfer learning (frozen base)
PHASE2_EPOCHS = 15          # Fine-tuning (unfrozen top layers)
PHASE1_LR = 1e-3
PHASE2_LR = 1e-5
FINE_TUNE_AT_LAYER = 100
VALIDATION_SPLIT = 0.15
MIN_IMAGES_PER_CLASS = 5    # Skip classes with fewer images

# === Regularization ===
DROPOUT_RATE = 0.3
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 1e-5

# === Callbacks ===
EARLY_STOPPING_PATIENCE = 5
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.5
MIN_LR = 1e-7

print("Config loaded.")
print(f"  Phase 1: {PHASE1_EPOCHS} epochs @ LR={PHASE1_LR}")
print(f"  Phase 2: {PHASE2_EPOCHS} epochs @ LR={PHASE2_LR}")
print(f"  Min images per class: {MIN_IMAGES_PER_CLASS}")

## 3. Auto-Discover & Merge All Datasets

This cell scans every dataset you attached via "+ Add Input" and merges them
into a single unified folder structure. It handles:
- **Folder-per-class** datasets (Indian Food, MAFood-121, Food-101, etc.)
- **Numbered folder** datasets (UECFood-256 with `category.txt` mapping)
- **Nested structures** (e.g., `dataset/images/train/class_name/`)
- **Duplicate class names** across datasets (merged into one class)
- Skips non-image files and classes below the minimum threshold

In [ ]:
def normalize_class_name(name):
    """Normalize a class name to a consistent format for merging across datasets."""
    name = name.strip().lower()
    # Replace separators with space
    for sep in ["_", "-", "."]:
        name = name.replace(sep, " ")
    # Remove extra whitespace
    name = " ".join(name.split())
    return name


def display_name(normalized):
    """Convert normalized name to Title Case for display."""
    return normalized.title()


def folder_name(normalized):
    """Convert normalized name to a safe folder name."""
    return normalized.replace(" ", "_")


def is_image(path):
    """Check if a file is an image by extension."""
    return Path(path).suffix.lower() in IMAGE_EXTENSIONS


def load_uec_category_map(dataset_path):
    """Load UECFood-256 category.txt mapping (folder_id -> class_name)."""
    cat_file = os.path.join(dataset_path, "category.txt")
    if not os.path.exists(cat_file):
        # Try one level deeper
        for root, dirs, files in os.walk(dataset_path):
            if "category.txt" in files:
                cat_file = os.path.join(root, "category.txt")
                break
    
    mapping = {}
    if os.path.exists(cat_file):
        with open(cat_file, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) >= 2:
                    mapping[parts[0].strip()] = parts[1].strip()
        print(f"    Loaded UEC category map: {len(mapping)} entries")
    return mapping


def find_image_class_dirs(root_path, depth=0, max_depth=4):
    """
    Recursively find directories that contain images (class folders).
    Returns list of (dir_path, class_name) tuples.
    """
    if depth > max_depth:
        return []
    
    results = []
    try:
        entries = sorted(os.listdir(root_path))
    except PermissionError:
        return []
    
    for entry in entries:
        entry_path = os.path.join(root_path, entry)
        if not os.path.isdir(entry_path):
            continue
        
        # Check if this directory directly contains images
        has_images = any(
            is_image(os.path.join(entry_path, f))
            for f in os.listdir(entry_path)[:20]  # Check first 20 files
            if os.path.isfile(os.path.join(entry_path, f))
        )
        
        if has_images:
            results.append((entry_path, entry))
        else:
            # Recurse deeper (handles nested structures like dataset/images/train/class/)
            results.extend(find_image_class_dirs(entry_path, depth + 1, max_depth))
    
    return results


def scan_and_merge_datasets():
    """
    Scan all datasets in /kaggle/input/ and create symlinks into a merged directory.
    Uses symlinks to avoid copying (saves disk space on Kaggle).
    """
    # Clean previous merge
    if os.path.exists(MERGED_DIR):
        shutil.rmtree(MERGED_DIR)
    os.makedirs(MERGED_DIR)
    
    # Stats
    class_image_count = defaultdict(int)
    dataset_stats = {}
    
    # Scan each dataset in /kaggle/input/
    if not os.path.exists(INPUT_DIR):
        print(f"ERROR: {INPUT_DIR} not found. Are you running on Kaggle?")
        return {}, []
    
    datasets = sorted(os.listdir(INPUT_DIR))
    if not datasets:
        print("ERROR: No datasets found! Click '+ Add Input' to add datasets.")
        return {}, []
    
    print(f"Found {len(datasets)} dataset(s) in {INPUT_DIR}:")
    
    for ds_name in datasets:
        ds_path = os.path.join(INPUT_DIR, ds_name)
        if not os.path.isdir(ds_path):
            continue
        
        print(f"\n  Scanning: {ds_name}")
        
        # Check for UECFood-256 style numbered folders
        uec_map = load_uec_category_map(ds_path)
        
        # Find all class directories
        class_dirs = find_image_class_dirs(ds_path)
        
        if not class_dirs:
            print(f"    No image class folders found, skipping.")
            continue
        
        ds_classes = 0
        ds_images = 0
        
        for dir_path, raw_name in class_dirs:
            # Resolve class name
            if uec_map and raw_name in uec_map:
                class_name = normalize_class_name(uec_map[raw_name])
            else:
                class_name = normalize_class_name(raw_name)
            
            # Skip numeric-only names without a mapping
            if class_name.isdigit() and not uec_map:
                continue
            
            safe_folder = folder_name(class_name)
            target_dir = os.path.join(MERGED_DIR, safe_folder)
            os.makedirs(target_dir, exist_ok=True)
            
            # Symlink images into merged directory
            img_count = 0
            for fname in os.listdir(dir_path):
                src = os.path.join(dir_path, fname)
                if os.path.isfile(src) and is_image(src):
                    # Prefix with dataset name to avoid filename collisions
                    dst_name = f"{ds_name}_{fname}"
                    dst = os.path.join(target_dir, dst_name)
                    if not os.path.exists(dst):
                        os.symlink(src, dst)
                        img_count += 1
            
            class_image_count[safe_folder] += img_count
            ds_images += img_count
            if img_count > 0:
                ds_classes += 1
        
        dataset_stats[ds_name] = {"classes": ds_classes, "images": ds_images}
        print(f"    Found {ds_classes} classes, {ds_images} images")
    
    # Remove classes with too few images
    removed = 0
    for cls_folder, count in list(class_image_count.items()):
        if count < MIN_IMAGES_PER_CLASS:
            shutil.rmtree(os.path.join(MERGED_DIR, cls_folder))
            del class_image_count[cls_folder]
            removed += 1
    
    # Get final class list
    class_names = sorted(class_image_count.keys())
    total_images = sum(class_image_count.values())
    
    print(f"\n{'='*60}")
    print(f"MERGED DATASET SUMMARY")
    print(f"{'='*60}")
    print(f"  Total classes:  {len(class_names)}")
    print(f"  Total images:   {total_images:,}")
    print(f"  Removed {removed} classes with < {MIN_IMAGES_PER_CLASS} images")
    print(f"\n  Per-dataset breakdown:")
    for ds, stats in dataset_stats.items():
        print(f"    {ds:45s} {stats['classes']:4d} classes, {stats['images']:6d} images")
    
    # Show image count distribution
    counts = sorted(class_image_count.values())
    print(f"\n  Images per class: min={counts[0]}, median={counts[len(counts)//2]}, max={counts[-1]}")
    
    return class_image_count, class_names


class_image_count, class_names = scan_and_merge_datasets()
NUM_CLASSES = len(class_names)
print(f"\nNUM_CLASSES = {NUM_CLASSES}")

## 4. Save Label Map

In [ ]:
# Save label map for TFLite metadata
with open(LABEL_MAP_PATH, "w") as f:
    for name in class_names:
        f.write(f"{display_name(name.replace('_', ' '))}\n")

print(f"Label map saved: {LABEL_MAP_PATH} ({NUM_CLASSES} classes)")
print(f"\nFirst 20 classes:")
for i, name in enumerate(class_names[:20]):
    count = class_image_count[name]
    print(f"  {i+1:3d}. {display_name(name.replace('_', ' ')):30s} ({count} images)")
if NUM_CLASSES > 20:
    print(f"  ... and {NUM_CLASSES - 20} more")

## 5. Build Data Pipeline

Loads from the merged directory using `image_dataset_from_directory`
with data augmentation for training and class weight balancing.

In [ ]:
# --- Load datasets from merged directory ---

print("Loading datasets from merged directory...")

# Training set (with validation split)
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    MERGED_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=42,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=None,  # We'll batch after augmentation
    label_mode="int",
    shuffle=True,
)

# Validation set
raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    MERGED_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=42,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=None,
    label_mode="int",
    shuffle=True,
)

# Verify class names match our label map order
loaded_class_names = raw_train_ds.class_names
assert loaded_class_names == class_names, "Class name mismatch!"
print(f"  Classes verified: {len(loaded_class_names)}")

In [ ]:
# --- Compute class weights to handle imbalanced datasets ---

total_images = sum(class_image_count.values())
class_weights = {}
for i, name in enumerate(class_names):
    count = class_image_count[name]
    # Inverse frequency weighting, capped to avoid extreme values
    weight = total_images / (NUM_CLASSES * count)
    class_weights[i] = min(weight, 10.0)  # Cap at 10x

weights_list = [class_weights[i] for i in range(NUM_CLASSES)]
print(f"Class weights: min={min(weights_list):.2f}, max={max(weights_list):.2f}, median={sorted(weights_list)[len(weights_list)//2]:.2f}")

In [ ]:
# --- Preprocessing & augmentation functions ---

AUTOTUNE = tf.data.AUTOTUNE

def normalize(image, label):
    """Normalize pixels to [0, 1]."""
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def one_hot(image, label):
    """One-hot encode the label."""
    return image, tf.one_hot(label, NUM_CLASSES)

# Data augmentation layer (applied only to training)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom((-0.15, 0.0)),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
], name="data_augmentation")

def augment(image, label):
    """Apply augmentation to a single image."""
    image = data_augmentation(image, training=True)
    return image, label

# --- Build final pipelines ---

train_ds = (
    raw_train_ds
    .shuffle(10000)
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .map(one_hot, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    raw_val_ds
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .map(one_hot, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f"Pipeline ready:")
print(f"  Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"  Val batches:   {tf.data.experimental.cardinality(val_ds).numpy()}")

## 6. Visualize Sample Training Images

In [ ]:
import matplotlib.pyplot as plt

sample_images, sample_labels = next(iter(train_ds))

fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i].numpy().clip(0, 1))
        label_idx = tf.argmax(sample_labels[i]).numpy()
        ax.set_title(display_name(class_names[label_idx].replace('_', ' ')), fontsize=9)
    ax.axis("off")

plt.suptitle("Sample Training Images (with augmentation)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Build the Model

EfficientNetV2B0 (pretrained ImageNet) + custom classification head.
Output size adapts automatically to however many food classes were discovered.

In [ ]:
from tensorflow.keras import layers, regularizers

def build_model(num_classes):
    """Build food classifier with EfficientNetV2B0 backbone."""
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
    
    base_model = tf.keras.applications.EfficientNetV2B0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        include_preprocessing=False,
    )
    base_model.trainable = False
    
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                     name="dense_1")(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_1")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                     name="dense_2")(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="food_classifier_v2")
    return model, base_model

model, base_model = build_model(NUM_CLASSES)

trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
total = model.count_params()
print(f"Model built for {NUM_CLASSES} food classes")
print(f"  Total params:     {total:,}")
print(f"  Trainable params: {trainable:,} (head only)")
print(f"  Base model frozen: {not base_model.trainable}")

## 8. Phase 1 — Transfer Learning (Frozen Base)

Train only the classification head. Uses class weights to handle
imbalanced datasets (some classes may have 50 images, others 750).

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)

phase1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "best_phase1.keras"),
        monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=EARLY_STOPPING_PATIENCE,
        mode="max", restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1,
    ),
]

print("=" * 60)
print(f"PHASE 1: Transfer Learning — {NUM_CLASSES} classes, frozen base")
print("=" * 60)

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weights,
    callbacks=phase1_callbacks,
)

print(f"\nPhase 1 complete!")
print(f"  Best val accuracy: {max(history_p1.history['val_accuracy'])*100:.2f}%")
print(f"  Best val top-5:    {max(history_p1.history['val_top5_accuracy'])*100:.2f}%")

## 9. Phase 2 — Fine-Tuning (Unfreeze Top Layers)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT_LAYER]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
frozen = sum(1 for l in base_model.layers if not l.trainable)
print(f"Fine-tuning: {unfrozen} unfrozen, {frozen} frozen layers")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)

phase2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "best_phase2.keras"),
        monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=EARLY_STOPPING_PATIENCE,
        mode="max", restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1,
    ),
]

print("\n" + "=" * 60)
print("PHASE 2: Fine-Tuning (top layers unfrozen)")
print("=" * 60)

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    class_weight=class_weights,
    callbacks=phase2_callbacks,
)

print(f"\nPhase 2 complete!")
print(f"  Best val accuracy: {max(history_p2.history['val_accuracy'])*100:.2f}%")
print(f"  Best val top-5:    {max(history_p2.history['val_top5_accuracy'])*100:.2f}%")

## 10. Training History Plots

In [ ]:
import matplotlib.pyplot as plt

def plot_history(h1, h2):
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]
    top5 = h1.history["top5_accuracy"] + h2.history["top5_accuracy"]
    val_top5 = h1.history["val_top5_accuracy"] + h2.history["val_top5_accuracy"]
    
    epochs = range(1, len(acc) + 1)
    phase1_end = len(h1.history["accuracy"])
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for ax, (train, val, title) in zip(axes, [
        (acc, val_acc, "Top-1 Accuracy"),
        (top5, val_top5, "Top-5 Accuracy"),
        (loss, val_loss, "Loss"),
    ]):
        ax.plot(epochs, train, "b-", label="Train")
        ax.plot(epochs, val, "r-", label="Validation")
        ax.axvline(x=phase1_end, color="gray", linestyle="--", alpha=0.7, label="Fine-tune start")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f"Training History — {NUM_CLASSES} Food Classes", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()

plot_history(history_p1, history_p2)

## 11. Evaluate on Validation Set

In [ ]:
print("Evaluating on validation set...")
val_results = model.evaluate(val_ds, verbose=1)

print(f"\n{'='*50}")
print(f"VALIDATION RESULTS ({NUM_CLASSES} classes)")
print(f"{'='*50}")
print(f"  Loss:           {val_results[0]:.4f}")
print(f"  Top-1 Accuracy: {val_results[1]*100:.2f}%")
print(f"  Top-5 Accuracy: {val_results[2]*100:.2f}%")

## 12. Per-Class Analysis

In [ ]:
from collections import Counter

all_preds = []
all_labels = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    all_preds.append(preds)
    all_labels.append(labels.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

pred_classes = np.argmax(all_preds, axis=1)
true_classes = np.argmax(all_labels, axis=1)

per_class = []
for i, name in enumerate(class_names):
    mask = true_classes == i
    if mask.sum() == 0:
        continue
    acc = np.mean(pred_classes[mask] == i)
    per_class.append((display_name(name.replace('_', ' ')), acc, int(mask.sum())))

per_class.sort(key=lambda x: x[1])

print("WORST 15 classes:")
print("-" * 55)
for name, acc, count in per_class[:15]:
    print(f"  {name:35s} {acc*100:5.1f}% ({count} samples)")

print(f"\nBEST 15 classes:")
print("-" * 55)
for name, acc, count in per_class[-15:]:
    print(f"  {name:35s} {acc*100:5.1f}% ({count} samples)")

# Most confused pairs
confusion_pairs = Counter()
for t, p in zip(true_classes, pred_classes):
    if t != p:
        confusion_pairs[(class_names[t], class_names[p])] += 1

print(f"\nMOST CONFUSED PAIRS:")
print("-" * 65)
for (tn, pn), count in confusion_pairs.most_common(15):
    t = display_name(tn.replace('_', ' '))
    p = display_name(pn.replace('_', ' '))
    print(f"  {t:28s} -> {p:28s} ({count}x)")

## 13. Export to TFLite

Float16 quantization — cuts model size ~50% with negligible accuracy loss.

In [ ]:
print("Converting to TFLite (float16 quantization)...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open(TFLITE_MODEL_PATH, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_MODEL_PATH) / (1024 * 1024)
print(f"\nTFLite model saved: {TFLITE_MODEL_PATH}")
print(f"Model size: {size_mb:.1f} MB")
print(f"Classes: {NUM_CLASSES}")

In [ ]:
# --- Embed metadata + labels into the TFLite model ---

try:
    from tflite_support.metadata_writers import image_classifier
    from tflite_support.metadata_writers import writer_utils

    writer = image_classifier.MetadataWriter.create_for_inference(
        writer_utils.load_file(TFLITE_MODEL_PATH),
        model_name="What's In My Thattu Food Classifier v2",
        model_description=(
            f"Food recognition model with {NUM_CLASSES} categories. "
            f"Trained using EfficientNetV2B0 transfer learning on multiple "
            f"combined food datasets for broad coverage."
        ),
        input_norm_mean=[0.0],
        input_norm_std=[255.0],
        label_file_paths=[LABEL_MAP_PATH],
    )

    writer_utils.save_file(writer.populate(), TFLITE_MODEL_PATH)
    print("Metadata + labels embedded into TFLite model.")
    print("Android's mlModelBinding will auto-generate the wrapper class.")

except Exception as e:
    print(f"Metadata embedding skipped: {e}")
    print("Saving labels.txt separately as fallback.")
    import shutil
    fallback = TFLITE_MODEL_PATH.replace(".tflite", "_labels.txt")
    shutil.copy(LABEL_MAP_PATH, fallback)
    print(f"Fallback labels: {fallback}")

## 14. Verify TFLite Model

In [ ]:
print("Verifying TFLite model...")

interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"  Input:  shape={input_details[0]['shape']}, dtype={input_details[0]['dtype']}")
print(f"  Output: shape={output_details[0]['shape']}, dtype={output_details[0]['dtype']}")

# Test with a real validation image
test_batch = next(iter(val_ds))
test_image = test_batch[0][0:1].numpy().astype(np.float32)
true_label = np.argmax(test_batch[1][0].numpy())

interpreter.set_tensor(input_details[0]["index"], test_image)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])[0]

top5_indices = np.argsort(output)[::-1][:5]
true_name = display_name(class_names[true_label].replace('_', ' '))

print(f"\n  True label: {true_name}")
print(f"  TFLite Top-5:")
for i, idx in enumerate(top5_indices, 1):
    name = display_name(class_names[idx].replace('_', ' '))
    print(f"    {i}. {name:30s} {output[idx]*100:.1f}%")

print(f"\n  Softmax sum: {output.sum():.4f}")
print(f"  Verification: PASSED")

## 15. TFLite Accuracy Spot-Check

In [ ]:
NUM_EVAL = 500
correct = correct_top5 = total = 0

for images, labels in val_ds:
    for i in range(images.shape[0]):
        if total >= NUM_EVAL:
            break
        img = images[i:i+1].numpy().astype(np.float32)
        true_label = np.argmax(labels[i].numpy())
        
        interpreter.set_tensor(input_details[0]["index"], img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])[0]
        
        if np.argmax(output) == true_label:
            correct += 1
        if true_label in np.argsort(output)[::-1][:5]:
            correct_top5 += 1
        total += 1
    if total >= NUM_EVAL:
        break

print(f"TFLite accuracy on {total} samples:")
print(f"  Top-1: {correct/total*100:.2f}%")
print(f"  Top-5: {correct_top5/total*100:.2f}%")

## 16. Save Everything & Summary

In [ ]:
# Save Keras model for future fine-tuning
keras_path = os.path.join(OUTPUT_DIR, "food_classifier_v2.keras")
model.save(keras_path)

# Save training report
report = {
    "num_classes": NUM_CLASSES,
    "class_names": [display_name(n.replace('_', ' ')) for n in class_names],
    "total_images": sum(class_image_count.values()),
    "phase1_best_val_acc": float(max(history_p1.history['val_accuracy'])),
    "phase2_best_val_acc": float(max(history_p2.history['val_accuracy'])),
    "val_top1_accuracy": float(val_results[1]),
    "val_top5_accuracy": float(val_results[2]),
    "tflite_size_mb": os.path.getsize(TFLITE_MODEL_PATH) / (1024 * 1024),
}
with open(os.path.join(OUTPUT_DIR, "training_report.json"), "w") as f:
    json.dump(report, f, indent=2)

tflite_mb = os.path.getsize(TFLITE_MODEL_PATH) / (1024 * 1024)
keras_mb = os.path.getsize(keras_path) / (1024 * 1024)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Food classes:       {NUM_CLASSES}")
print(f"  Total images:       {sum(class_image_count.values()):,}")
print(f"  Phase 1 best val:   {max(history_p1.history['val_accuracy'])*100:.2f}%")
print(f"  Phase 2 best val:   {max(history_p2.history['val_accuracy'])*100:.2f}%")
print(f"  Final val top-1:    {val_results[1]*100:.2f}%")
print(f"  Final val top-5:    {val_results[2]*100:.2f}%")
print(f"")
print(f"  OUTPUT FILES (download from Output tab):")
print(f"    whats_in_my_thattu_v2.tflite   ({tflite_mb:.1f} MB) <- Deploy to Android")
print(f"    labels.txt                      ({NUM_CLASSES} classes)")
print(f"    food_classifier_v2.keras         ({keras_mb:.1f} MB)")
print(f"    training_report.json")
print(f"    training_curves.png")

## 17. Deploy to Android

### Step 1: Download
Download `whats_in_my_thattu_v2.tflite` from the **Output** tab.

### Step 2: Copy to project
```bash
cp whats_in_my_thattu_v2.tflite \
  tensorImageInterpreter/src/main/ml/whats_in_my_thattu_v2.tflite
```

### Step 3: Update the model reference
In `TensorImageInterpreter.kt`, Android's `mlModelBinding` auto-generates
a class from the filename. Update the import and constructor:

```kotlin
// Change:
import com.dineshworkspace.tensorimageinterpreter.ml.WhatsInMyThattu
private val model: WhatsInMyThattu = WhatsInMyThattu.newInstance(context)

// To:
import com.dineshworkspace.tensorimageinterpreter.ml.WhatsInMyThattuV2
private val model: WhatsInMyThattuV2 = WhatsInMyThattuV2.newInstance(context)
```

### Step 4: Rebuild
The `TensorImageInterpreter` already preprocesses correctly (resize 224x224 + normalize to [0,1]).
The metadata-embedded labels will be returned as `displayName` in `FoodMatch`.

### Want even more food categories?
Add more datasets in Kaggle, re-run this notebook. The model output layer
auto-sizes to however many classes are found. The Android side doesn't need
any changes since labels are embedded in the `.tflite` metadata.